# 传感器扫描数据分析 - 衍射效率计算与热力图

本 Notebook 用于解析 `.strc` (HDF5) 格式的传感器扫描数据文件，
计算衍射效率并生成热力图，支持差分热力图分析。

## 使用说明
1. **Cell 1**: 配置参数 — 修改文件路径、传感器配对、效率系数等
2. **Cell 2**: 加载数据 — 读取 HDF5 文件并预览
3. **Cell 3**: 衍射效率热力图 — 选择光斑，计算并绘制热力图
4. **Cell 4**: 差分热力图 — 选择两个光斑，设置偏移量，计算差分

## 1. 配置参数

根据实际情况修改以下参数。

In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

from config import AppConfig, SpotConfig
from data_loader import load_strc_file, get_file_metadata
from analysis import compute_diffraction_efficiency, build_grid, compute_differential
from visualization import plot_heatmap_mpl, plot_diff_heatmap_mpl

# ========================
# 在这里修改配置参数
# ========================

# 数据文件路径
DATA_FILE = "20251023_080353.strc"

# 传感器配对关系配置
# 每个 SpotConfig 定义一个光斑：
#   name: 光斑名称
#   incident_channel: 入射光通道编号 (1~12)
#   reflected_channel: 反射光通道编号 (1~12)
#   incident_efficiency: 入射光效率系数 (默认 1.0)
#   reflected_efficiency: 反射光效率系数 (默认 1.0)
config = AppConfig(
    spots=[
        SpotConfig(name="光斑1", incident_channel=1, reflected_channel=2,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
        SpotConfig(name="光斑2", incident_channel=3, reflected_channel=4,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
        SpotConfig(name="光斑3", incident_channel=5, reflected_channel=9,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
        SpotConfig(name="光斑4", incident_channel=6, reflected_channel=10,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
        SpotConfig(name="光斑5", incident_channel=7, reflected_channel=11,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
        SpotConfig(name="光斑6", incident_channel=8, reflected_channel=12,
                   incident_efficiency=1.0, reflected_efficiency=1.0),
    ],
    # 衍射效率合格范围
    pass_range=(0.5, 1.0),
)

print("配置完成！")
print(f"数据文件: {DATA_FILE}")
print(f"光斑数量: {len(config.spots)}")
for s in config.spots:
    print(f"  {s.name}: 入射通道={s.incident_channel}(效率={s.incident_efficiency}), "
          f"反射通道={s.reflected_channel}(效率={s.reflected_efficiency})")
print(f"合格范围: {config.pass_range}")

## 2. 加载数据并预览

In [ ]:
# 读取文件元数据
metadata = get_file_metadata(DATA_FILE, config)
print("文件元数据:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

# 加载数据到 DataFrame
print("\n正在加载数据...")
df = load_strc_file(DATA_FILE, config)
print(f"数据加载完成！共 {len(df)} 行, {len(df.columns)} 列")
print(f"X 范围: [{df['x'].min():.2f}, {df['x'].max():.2f}]")
print(f"Y 范围: [{df['y'].min():.2f}, {df['y'].max():.2f}]")

# 显示前几行数据
print("\n数据预览:")
df.head(10)

## 3. 衍射效率热力图

选择一个光斑，计算衍射效率并绘制热力图。  
修改下方的 `SPOT_NAME` 来切换不同的光斑。

In [ ]:
# ========================
# 在这里选择要分析的光斑
# ========================
SPOT_NAME = "光斑1"  # 可选: 光斑1 ~ 光斑6

# 获取光斑配置
spot = config.get_spot_by_name(SPOT_NAME)
if spot is None:
    raise ValueError(f"未找到光斑 '{SPOT_NAME}'，可用: {config.get_spot_names()}")

print(f"分析光斑: {spot.name}")
print(f"  入射光: 通道{spot.incident_channel} (效率系数={spot.incident_efficiency})")
print(f"  反射光: 通道{spot.reflected_channel} (效率系数={spot.reflected_efficiency})")

# 计算衍射效率
de = compute_diffraction_efficiency(df, spot)
print(f"\n衍射效率统计:")
print(f"  最小值: {de[~__import__('numpy').isnan(de)].min():.4f}")
print(f"  最大值: {de[~__import__('numpy').isnan(de)].max():.4f}")
print(f"  平均值: {de[~__import__('numpy').isnan(de)].mean():.4f}")

# 构建二维网格
print("\n正在构建网格...")
x_grid, y_grid, grid_2d = build_grid(
    df['x'].values, df['y'].values, de,
    resolution=config.grid_resolution,
)
print(f"网格大小: {grid_2d.shape} (Y × X)")

# 绘制热力图
fig = plot_heatmap_mpl(
    x_grid, y_grid, grid_2d,
    title=f"{spot.name} 衍射效率",
    pass_range=config.pass_range,
)
fig.show()

## 4. 差分热力图

选择两个光斑，输入位置偏移 (dx, dy)，计算差分并绘制。  
差分 = 光斑A的衍射效率 - 光斑B的衍射效率（偏移对齐后）

In [ ]:
# ========================
# 在这里配置差分参数
# ========================
SPOT_A_NAME = "光斑1"  # 第一个光斑
SPOT_B_NAME = "光斑2"  # 第二个光斑
DX = 0.0               # X 方向位置偏移 (mm)
DY = 0.0               # Y 方向位置偏移 (mm)

# 获取配置
spot_a = config.get_spot_by_name(SPOT_A_NAME)
spot_b = config.get_spot_by_name(SPOT_B_NAME)
if spot_a is None:
    raise ValueError(f"未找到光斑 '{SPOT_A_NAME}'，可用: {config.get_spot_names()}")
if spot_b is None:
    raise ValueError(f"未找到光斑 '{SPOT_B_NAME}'，可用: {config.get_spot_names()}")

print(f"差分分析: {spot_a.name} - {spot_b.name}")
print(f"位置偏移: dx={DX}, dy={DY}")

# 计算差分
print("\n正在计算差分（可能需要一些时间）...")
x_diff, y_diff, diff_grid = compute_differential(
    df, spot_a, spot_b, dx=DX, dy=DY,
)

import numpy as np
valid = diff_grid[~np.isnan(diff_grid)]
print(f"差分网格大小: {diff_grid.shape}")
print(f"差分统计:")
print(f"  最小值: {valid.min():.4f}")
print(f"  最大值: {valid.max():.4f}")
print(f"  平均值: {valid.mean():.4f}")
print(f"  标准差: {valid.std():.4f}")

# 绘制差分热力图
fig = plot_diff_heatmap_mpl(
    x_diff, y_diff, diff_grid,
    title=f"差分: {spot_a.name} - {spot_b.name} (偏移 dx={DX}, dy={DY})",
)
fig.show()